# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baselsalah342-max/flyrank_intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*



My lane (Refresh / Content Opportunity Scoring) is primarily a **classification** task at its
core: I am predicting a binary label — `is_declining_label` (1 = trend_direction == "down",
0 = otherwise) — for each content page.

But the deliverable my research question asks for is a **ranking/scoring** output: an ordered
refresh queue that tells an editor which pages to review first. I get there by training a
classifier to output a probability (not just a hard 0/1), then sorting pages by that probability.
This is exactly what the week-1 pipeline did — Precision@50 measures ranking quality on top of
a classifier's output.

So: classification is the underlying task type, and scoring/ranking is how the output gets used
to support the "which ones first" decision.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is `is_declining_label`, a binary label already engineered by the pipeline:
1 when `trend_direction == "down"` (last-30d impressions fell more than 20% vs prev-30d),
0 otherwise. This is 16,262 of 30,000 rows (54.2%).

This is an OBSERVED outcome, not a rule I'm defining myself — it comes from actual measured
impression changes over time, which satisfies the "target must be observed" rule.

Important leakage trap: since the label is built from `trend_direction` and `trend_pct`,
neither of those two columns (nor anything derived from them) can ever be used as a model
feature — that would let the model see the answer. My features must come from the other
~40+ columns (position, CTR, content_type, word_count, content_age_days, etc.), not from
the trend columns themselves.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My success metric is Precision@50: of the top 50 pages the model ranks highest for refresh
priority, what fraction are actually declining (is_declining_label == 1)?

This fits the decision because editors have limited capacity — they can't review all ~30,000
pages, only the top of a queue. Precision@K measures whether the model is right about exactly
the pages someone will act on, not overall accuracy across all rows (which would be misleading
here since ~54% of the data is already labeled "declining").

Baseline from week 1: a hand-written rule scored 0.240 Precision@50, while a random forest
scored 0.740 — roughly 3x better. "Good" for my lane means beating that baseline by a
meaningful margin on a held-out, client-grouped split (not just fitting the training data).

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page (identified by content_id, within one client_id), summarizing its
trailing-90-day performance. My lane uses the full starter dataset — all 30,000 rows across
32 clients — since I'm scoring every page for refresh priority, not filtering to one segment.

Below: content_id/client_id (unit identifiers, never features), avg_position/ctr/content_type/
word_count/content_age_days (candidate features), and is_declining_label (the target) —
showing a real slice so the grain is provable, not just claimed.


In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# prepare the same target the pipeline uses, without touching feature columns
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

cols = ["content_id", "client_id", "avg_position", "ctr", "content_type",
        "word_count", "content_age_days", "is_declining_label"]

df[cols].head(8)

,content_id,client_id,avg_position,ctr,content_type,word_count,content_age_days,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.6,0.76,keyword article,3221.0,187,1
1,content_a1fb4e703a9e,client_4e07408562,20.3,0.05,keyword article,2481.0,445,1
2,content_9aa793d4d895,client_7f2253d7e2,36.5,0.09,keyword article,3515.0,141,1
3,content_331d6c4de07b,client_19581e27de,6.2,0.49,keyword article,NaN,463,0
4,content_d99b7a2d90ca,client_3fdba35f04,44.0,0.13,keyword article,2803.0,263,1
5,content_d4084a4bc775,client_f369cb89fc,8.5,0.03,keyword article,3080.0,147,1
6,content_9a34b442b552,client_8722616204,7.0,0.00,keyword article,3059.0,90,1
7,content_a63219c6e95a,client_19581e27de,21.2,0.06,keyword article,NaN,445,0


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule isn't enough because refresh-worthiness depends on several signals interacting
at once — avg_position, ctr, content_type, word_count, and content_age_days together — not
any single threshold. A rule like "flag pages below position 10" ignores that the same position
means different things for different content_types, and can't weigh combinations the way a
model can.

This isn't just a theoretical claim — week 1's pipeline tested it directly: a hand-written
baseline rule scored 0.240 Precision@50, while a random forest scored 0.740 on the same data
and split — about 3.1x better. That gap is the evidence that the pattern is real but too
tangled for an if-statement to capture by hand.

Hand-written rule  Precision@50: 0.240
Random forest      Precision@50: 0.740   (~3.1x better)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.